# MSPR Municipales 2032 — Partie Data Scientist

**Objectif :** préparer les variables ML, tester la prise en compte de `covid_2020` et produire une première projection exploratoire pour 2032.

**Livrables couverts dans ce notebook :**
- notebook exécutable ;
- préparation des variables ML ;
- résultats comparatifs avec / sans traitement spécifique de 2020 ;
- projection exploratoire 2032 ;
- limites du modèle.

> Remarque : ce notebook utilise les liens `explore.data.gouv.fr` fournis dans le sujet et l'API tabulaire data.gouv.fr. Si l'API est lente ou indisponible, téléchargez les CSV depuis l'explorateur puis placez-les dans `data/`.

In [ ]:
# Installation éventuelle dans Google Colab
# !pip install pandas numpy scikit-learn matplotlib requests -q

import re
import json
import math
import warnings
from pathlib import Path
from urllib.parse import urlparse, parse_qs

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

DATA_DIR = Path('data')
OUT_DIR = Path('outputs')
FIG_DIR = OUT_DIR / 'figures'
for p in [DATA_DIR, OUT_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Environnement prêt')

## 1. Liens des données

Les deux liens fournis pointent vers la même ressource agrégée, filtrée par `id_election` :
- `2020_muni_t1` pour les municipales 2020 premier tour ;
- `2014_muni_t1` pour les municipales 2014 premier tour.

Pour une vraie projection 2032, il faut idéalement ajouter 2026 quand le fichier est disponible : `2026_muni_t1`.

In [ ]:
URLS_ELECTIONS = {
    2014: 'https://explore.data.gouv.fr/fr/datasets/6481e741d4cf002ec0efec9d/?id_election__exact=2014_muni_t1#/resources/b8703c69-a18f-46ab-9e7f-3a8368dcb891',
    2020: 'https://explore.data.gouv.fr/fr/datasets/6481e741d4cf002ec0efec9d/?id_election__exact=2020_muni_t1#/resources/b8703c69-a18f-46ab-9e7f-3a8368dcb891',
    # Décommentez si vous avez le filtre 2026 disponible dans l'explorateur
    # 2026: 'https://explore.data.gouv.fr/fr/datasets/6481e741d4cf002ec0efec9d/?id_election__exact=2026_muni_t1#/resources/b8703c69-a18f-46ab-9e7f-3a8368dcb891',
}

RESOURCE_ID = 'b8703c69-a18f-46ab-9e7f-3a8368dcb891'
API_BASE = 'https://tabular-api.data.gouv.fr/api/resources'

## 2. Chargement depuis l'API tabulaire

Cette fonction récupère les lignes filtrées par année d'élection. Elle pagine automatiquement si nécessaire.

In [ ]:
def extract_id_election(url: str) -> str:\n    match = re.search(r'id_election__exact=([^#&]+)', url)\n    if not match:\n        raise ValueError(f'Impossible de trouver id_election dans : {url}')\n    return match.group(1)\n\ndef load_election_from_api(year: int, url: str, page_size: int = 5000, max_pages: int | None = None) -> pd.DataFrame:\n    id_election = extract_id_election(url)\n    rows = []\n    page = 1\n    while True:\n        params = {\n            'id_election__exact': id_election,\n            'page': page,\n            'page_size': page_size,\n        }\n        endpoint = f'{API_BASE}/{RESOURCE_ID}/data/'\n        r = requests.get(endpoint, params=params, timeout=60)\n        r.raise_for_status()\n        payload = r.json()\n        batch = payload.get('data', [])\n        rows.extend(batch)\n        total = payload.get('meta', {}).get('total', len(rows))\n        print(f'Année {year} | page {page} | lignes cumulées {len(rows)}/{total}')\n        if len(rows) >= total or not payload.get('links', {}).get('next'):\n            break\n        page += 1\n        if max_pages is not None and page > max_pages:\n            break\n    df = pd.DataFrame(rows)\n    df['annee'] = year\n    df['id_election'] = id_election\n    return df\n\n# Chargement réel\ndfs = []\nfor year, url in URLS_ELECTIONS.items():\n    df_year = load_election_from_api(year, url)\n    dfs.append(df_year)\n\nraw = pd.concat(dfs, ignore_index=True)\nprint(raw.shape)\nraw.head()

## 3. Harmonisation des colonnes et agrégation commune

Objectif : construire un panel propre par commune et par année avec les cibles :
- `nombre_votants` ;
- `taux_participation`.

In [ ]:
def clean_column_name(c: str) -> str:\n    c = str(c).strip().lower()\n    replacements = {'é':'e','è':'e','ê':'e','à':'a','â':'a','ù':'u','ç':'c','ï':'i','î':'i','ô':'o'}\n    for a,b in replacements.items():\n        c = c.replace(a,b)\n    c = re.sub(r'[^a-z0-9]+', '_', c).strip('_')\n    return c\n\ndef find_col(df, candidates):\n    cols = list(df.columns)\n    for cand in candidates:\n        if cand in cols:\n            return cand\n    for cand in candidates:\n        for col in cols:\n            if cand in col:\n                return col\n    return None\n\ndef to_num(s):\n    return pd.to_numeric(\n        s.astype(str).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),\n        errors='coerce'\n    )\n\ndef prepare_election_panel(raw: pd.DataFrame) -> pd.DataFrame:\n    df = raw.copy()\n    df.columns = [clean_column_name(c) for c in df.columns]\n    \n    col_code = find_col(df, ['code_commune', 'code_commune_insee', 'commune_code', 'code_insee', 'code'])\n    col_nom = find_col(df, ['nom_commune', 'commune', 'libelle_commune', 'libelle'])\n    col_inscrits = find_col(df, ['inscrits', 'nombre_d_inscrit', 'nombre_inscrit', 'nb_inscrits'])\n    col_votants = find_col(df, ['votants', 'nombre_de_votants', 'nb_votants'])\n    col_abst = find_col(df, ['abstentions', 'nombre_abstentions', 'nb_abstentions'])\n    col_expr = find_col(df, ['exprimes', 'suffrages_exprimes', 'nombre_de_suffrage', 'nombre_suffrage'])\n    \n    required = {'code_commune': col_code, 'nom_commune': col_nom, 'inscrits': col_inscrits, 'votants': col_votants}\n    print('Colonnes détectées :', required)\n    missing = [k for k,v in required.items() if v is None]\n    if missing:\n        raise ValueError('Colonnes indispensables non détectées : ' + ', '.join(missing))\n    \n    keep = ['annee']\n    rename = {col_code:'code_commune', col_nom:'nom_commune', col_inscrits:'nombre_inscrits', col_votants:'nombre_votants'}\n    if col_abst: rename[col_abst] = 'nombre_abstentions'\n    if col_expr: rename[col_expr] = 'nombre_exprimes'\n    use_cols = list(rename.keys()) + keep\n    out = df[use_cols].rename(columns=rename)\n    \n    out['code_commune'] = out['code_commune'].astype(str).str.extract(r'(\d+)')[0].str.zfill(5)\n    for c in ['nombre_inscrits','nombre_votants','nombre_abstentions','nombre_exprimes']:\n        if c in out.columns:\n            out[c] = to_num(out[c])\n    \n    agg = {'nom_commune':'first', 'nombre_inscrits':'sum', 'nombre_votants':'sum'}\n    if 'nombre_abstentions' in out.columns: agg['nombre_abstentions'] = 'sum'\n    if 'nombre_exprimes' in out.columns: agg['nombre_exprimes'] = 'sum'\n    panel = out.groupby(['code_commune','annee'], as_index=False).agg(agg)\n    panel['taux_participation'] = np.where(panel['nombre_inscrits'] > 0, panel['nombre_votants'] / panel['nombre_inscrits'] * 100, np.nan)\n    panel['taux_abstention'] = 100 - panel['taux_participation']\n    panel['covid_2020'] = (panel['annee'] == 2020).astype(int)\n    panel['periode_post_covid'] = (panel['annee'] > 2020).astype(int)\n    return panel\n\npanel = prepare_election_panel(raw)\npanel.to_csv(OUT_DIR / 'dataset_ml_municipales.csv', index=False, encoding='utf-8-sig')\nprint(panel.shape)\npanel.head()

## 4. Contrôle de l'effet Covid-19 en 2020

On compare la participation moyenne par année. Si 2020 est clairement plus basse, on justifie l'ajout de `covid_2020` ou une pondération plus faible de 2020.

In [ ]:
resume_annee = panel.groupby('annee').agg(
    communes=('code_commune', 'nunique'),
    inscrits_total=('nombre_inscrits', 'sum'),
    votants_total=('nombre_votants', 'sum'),
    participation_moyenne=('taux_participation', 'mean'),
    participation_mediane=('taux_participation', 'median')
).reset_index()

resume_annee['participation_nationale_ponderee'] = resume_annee['votants_total'] / resume_annee['inscrits_total'] * 100
resume_annee.to_csv(OUT_DIR / 'resultats_resume_par_annee.csv', index=False, encoding='utf-8-sig')
resume_annee

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(resume_annee['annee'], resume_annee['participation_moyenne'], marker='o')
plt.title('Taux de participation moyen par année')
plt.xlabel('Année')
plt.ylabel('Participation moyenne (%)')
plt.grid(True, alpha=0.3)
plt.savefig(FIG_DIR / 'participation_moyenne_par_annee.png', dpi=150, bbox_inches='tight')
plt.show()

if 2020 in resume_annee['annee'].values:
    part_2020 = float(resume_annee.loc[resume_annee['annee']==2020, 'participation_moyenne'].iloc[0])
    autres = resume_annee.loc[resume_annee['annee']!=2020, 'participation_moyenne'].mean()
    print(f'Ecart 2020 vs autres années disponibles : {part_2020 - autres:.2f} points')

## 5. Préparation des variables ML

Variables explicatives utilisées :
- `annee` : tendance temporelle ;
- `nombre_inscrits` : taille du corps électoral ;
- `taux_abstention` uniquement pour prédire les votants, pas pour prédire la participation car c'est une transformation directe ;
- `covid_2020` : variable binaire pour isoler l'année atypique.

On évite les variables qui créent une fuite directe de la cible.

In [ ]:
def get_features(target):
    base = ['annee', 'nombre_inscrits', 'covid_2020', 'periode_post_covid']
    if target == 'nombre_votants' and 'taux_abstention' in panel.columns:
        base.append('taux_abstention')
    return [c for c in base if c in panel.columns and c != target]

for target in ['nombre_votants', 'taux_participation']:
    print(target, '=>', get_features(target))

## 6. Comparaison des scénarios

Trois scénarios sont testés :
1. `avec_2020_normal` : 2020 est utilisée comme une observation normale ;
2. `avec_covid_2020` : on garde 2020 mais on donne au modèle une variable `covid_2020` ;
3. `sans_2020` : on retire 2020 pour voir si la projection change fortement.

Le but n'est pas de promettre une prédiction parfaite, mais de vérifier si le traitement de 2020 change les résultats.

In [ ]:
def make_model(model_name):\n    if model_name == 'ridge':\n        return Pipeline([\n            ('imputer', SimpleImputer(strategy='median')),\n            ('scaler', StandardScaler()),\n            ('model', Ridge(alpha=1.0))\n        ])\n    if model_name == 'random_forest':\n        return Pipeline([\n            ('imputer', SimpleImputer(strategy='median')),\n            ('model', RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=5))\n        ])\n    raise ValueError(model_name)\n\ndef evaluate_scenario(data, target, scenario, model_name):\n    df = data.dropna(subset=[target]).copy()\n    if scenario == 'sans_2020':\n        df = df[df['annee'] != 2020].copy()\n    features = get_features(target)\n    if scenario == 'avec_2020_normal':\n        features = [f for f in features if f != 'covid_2020']\n    if len(df) < 20:\n        return None, None\n    X = df[features]\n    y = df[target]\n    test_size = 0.25 if len(df) >= 100 else 0.35\n    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)\n    model = make_model(model_name)\n    model.fit(X_train, y_train)\n    pred = model.predict(X_test)\n    rmse = mean_squared_error(y_test, pred, squared=False)\n    mae = mean_absolute_error(y_test, pred)\n    r2 = r2_score(y_test, pred) if len(y_test) > 1 else np.nan\n    row = {\n        'target': target,\n        'scenario': scenario,\n        'modele': model_name,\n        'nb_lignes': len(df),\n        'features': ', '.join(features),\n        'RMSE': rmse,\n        'MAE': mae,\n        'R2': r2,\n    }\n    fitted_full = make_model(model_name)\n    fitted_full.fit(X, y)\n    bundle = {'target': target, 'scenario': scenario, 'modele': model_name, 'features': features, 'model': fitted_full}\n    return row, bundle\n\nscenarios = ['avec_2020_normal', 'avec_covid_2020', 'sans_2020']\nmodels = ['ridge', 'random_forest']\nall_results = []\nbundles = []\n\nfor target in ['nombre_votants', 'taux_participation']:\n    for scenario in scenarios:\n        for model_name in models:\n            row, bundle = evaluate_scenario(panel, target, scenario, model_name)\n            if row is not None:\n                all_results.append(row)\n                bundles.append(bundle)\n\nresults = pd.DataFrame(all_results).sort_values(['target','MAE'])\nresults.to_csv(OUT_DIR / 'resultats_comparatifs_modeles.csv', index=False, encoding='utf-8-sig')\nresults

## 7. Projection exploratoire 2032

Méthode simple :
- on part de la dernière année disponible par commune ;
- on fixe `annee = 2032` ;
- on met `covid_2020 = 0` ;
- on garde les inscrits au dernier niveau connu par défaut.

Cette projection est exploratoire : pour une version plus réaliste, il faudra ajouter une projection démographique / inscrits.

In [ ]:
def choose_best_bundle(target, preferred_scenario='avec_covid_2020'):\n    subset = results[(results['target']==target) & (results['scenario']==preferred_scenario)].sort_values('MAE')\n    if subset.empty:\n        subset = results[results['target']==target].sort_values('MAE')\n    best = subset.iloc[0]\n    for b in bundles:\n        if b['target']==best['target'] and b['scenario']==best['scenario'] and b['modele']==best['modele']:\n            return b, best\n    raise RuntimeError('Modèle introuvable')\n\ndef make_projection_2032(panel):\n    last = panel.sort_values('annee').groupby('code_commune', as_index=False).tail(1).copy()\n    last['annee_reference'] = last['annee']\n    last['annee'] = 2032\n    last['covid_2020'] = 0\n    last['periode_post_covid'] = 1\n    return last\n\nprojection = make_projection_2032(panel)\n\nfor target in ['nombre_votants','taux_participation']:\n    bundle, best_row = choose_best_bundle(target, preferred_scenario='avec_covid_2020')\n    pred = bundle['model'].predict(projection[bundle['features']])\n    if target == 'taux_participation':\n        pred = np.clip(pred, 0, 100)\n    if target == 'nombre_votants':\n        pred = np.maximum(pred, 0)\n    projection[f'projection_2032_{target}'] = pred\n    projection[f'modele_{target}'] = f"{bundle['modele']} | {bundle['scenario']}"\n\ncols = [c for c in [\n    'code_commune','nom_commune','annee_reference','annee','nombre_inscrits',\n    'projection_2032_nombre_votants','projection_2032_taux_participation',\n    'modele_nombre_votants','modele_taux_participation'\n] if c in projection.columns]\n\nprojection_finale = projection[cols].copy()\nprojection_finale.to_csv(OUT_DIR / 'projection_exploratoire_2032.csv', index=False, encoding='utf-8-sig')\nprojection_finale.head(20)

In [ ]:
synthese_projection = pd.DataFrame({
    'indicateur': [
        'communes_projetees',
        'votants_2032_total_projete',
        'participation_2032_moyenne_projete',
        'participation_2032_mediane_projete'
    ],
    'valeur': [
        projection_finale['code_commune'].nunique(),
        projection_finale['projection_2032_nombre_votants'].sum() if 'projection_2032_nombre_votants' in projection_finale else np.nan,
        projection_finale['projection_2032_taux_participation'].mean() if 'projection_2032_taux_participation' in projection_finale else np.nan,
        projection_finale['projection_2032_taux_participation'].median() if 'projection_2032_taux_participation' in projection_finale else np.nan,
    ]
})
synthese_projection.to_csv(OUT_DIR / 'synthese_projection_2032.csv', index=False, encoding='utf-8-sig')
synthese_projection

## 8. Interprétation à mettre dans le rendu

### Résultats comparatifs
À compléter après exécution avec les valeurs obtenues dans `resultats_comparatifs_modeles.csv`.

Phrase utilisable :

> Nous avons comparé trois stratégies : traiter 2020 comme une année normale, ajouter une variable `covid_2020`, et exclure 2020. L'objectif était de vérifier si l'année 2020 modifie fortement la trajectoire apprise par le modèle. La stratégie retenue est celle qui obtient la meilleure erreur moyenne tout en restant interprétable.

### Projection 2032

> La projection 2032 est exploratoire. Elle utilise les dernières valeurs disponibles par commune et fixe `covid_2020 = 0`, car 2032 est supposée être une année électorale normale. Les résultats ne doivent pas être lus comme une prédiction définitive, mais comme une première estimation pour comparer les communes et préparer la soutenance.

### Limites du modèle

- Il y a peu d'années d'observation : 2014, 2020 et éventuellement 2026.
- 2020 est une année atypique liée au contexte sanitaire, donc elle peut biaiser la tendance.
- Les inscrits 2032 ne sont pas réellement connus : ici ils sont conservés au dernier niveau disponible.
- Le modèle n'intègre pas encore toutes les variables socio-économiques : population, chômage, revenus, pauvreté, sécurité, contexte politique local.
- Les résultats dépendent fortement de la qualité des jointures communales.

### Fichiers générés

- `outputs/dataset_ml_municipales.csv`
- `outputs/resultats_resume_par_annee.csv`
- `outputs/resultats_comparatifs_modeles.csv`
- `outputs/projection_exploratoire_2032.csv`
- `outputs/synthese_projection_2032.csv`
- `outputs/figures/participation_moyenne_par_annee.png`